# Multimodal AI — Blood Work Analysis

In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langchain.tools import tool
from langchain.agents import create_agent
import base64

load_dotenv()

C:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
C:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## Encode the image and send to the vision model

In [2]:
with open("blood_work.png", "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode()

llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct")

message = HumanMessage(content=[
    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
    {"type": "text",      "text": "This is a blood work report. Extract all test results and flag any values outside the normal range."}
])

response = llm.invoke([message])
print(response.content)

**Patient:** Rajesh Sharma  
**Age:** 48  
**Gender:** Male  
**Date:** May 7, 2026  

### **COMPLETE BLOOD COUNT (CBC)**  
- **Hemoglobin:** 15.1 g/dL **(Normal: 13.5-17.5) - WITHIN NORMAL RANGE**  
- **Hematocrit:** 44% **(Normal: 41-53) - WITHIN NORMAL RANGE**  
- **WBC:** 6.8 × 10^3/uL **(Normal: 4.5-11.0) - WITHIN NORMAL RANGE**  
- **Platelets:** 220 × 10^3/uL **(Normal: 150-400) - WITHIN NORMAL RANGE**  

### **LIPID PANEL**  
- **Total Cholesterol:** 238 mg/dL **(Normal: <200) - ELEVATED**  
- **LDL Cholesterol:** 162 mg/dL **(Normal: <100) - ELEVATED**  
- **HDL Cholesterol:** 36 mg/dL **(Normal: >40) - BELOW NORMAL**  
- **Triglycerides:** 188 mg/dL **(Normal: <150) - ELEVATED**  

### **METABOLIC PANEL**  
- **Glucose (Fasting):** 92 mg/dL **(Normal: 70-99) - WITHIN NORMAL RANGE**  
- **HbA1c:** 5.3% **(Normal: <5.7) - WITHIN NORMAL RANGE**  
- **Creatinine:** 1.0 mg/dL **(Normal: 0.7-1.3) - WITHIN NORMAL RANGE**  
- **eGFR:** 82 mL/min **(Normal: >60) - WITHIN NORMAL RANGE*

## Suggest Diet Plan Agent

The agent reads the blood work image, categorises the condition, then calls the diet tool.

In [3]:
@tool
def get_diet_recommendation(condition: str) -> dict:
    """Given a health condition, returns a diet plan. Condition must be one of: normal, high_cholesterol, high_sugar."""
    diet_plans = {
        "high_cholesterol": {
            "eat":        ["fruits", "vegetables", "whole grains", "lean protein"],
            "do_not_eat": ["red meat", "fried food", "full-fat dairy", "processed snacks"],
        },
        "high_sugar": {
            "eat":        ["vegetables", "whole grains", "legumes", "nuts"],
            "do_not_eat": ["white rice", "white sugar", "junk food", "sugary drinks"],
        },
        "normal": {
            "eat":        ["vegetables", "fruits", "whole grains", "lean protein"],
            "do_not_eat": ["excessive sugar", "processed food", "trans fats"],
        },
    }
    return diet_plans.get(condition, diet_plans["normal"])

In [4]:
SYSTEM_PROMPT = """
You are a helpful medical and nutrition assistant.
For the input blood work image, extract the numbers and the normal range, then categorize
the condition as one of: normal, high_cholesterol, high_sugar.
Then call the appropriate tool to retrieve and present the diet plan.
"""

diet_agent = create_agent(
    llm,
    tools=[get_diet_recommendation],
    system_prompt=SYSTEM_PROMPT,
)

In [5]:
result = diet_agent.invoke({
    "messages": [HumanMessage(content=[
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
        {"type": "text",      "text": "Analyse this blood work report and suggest a diet plan."},
    ])]
})

print(result["messages"][-1].content)

Based on the blood work report, the patient has high cholesterol. The diet plan for high cholesterol includes eating fruits, vegetables, whole grains, and lean protein, while avoiding red meat, fried food, full-fat dairy, and processed snacks.
